# 📖 Notebook 4: Helm & Kustomize — Managing YAML at Scale

In the last notebook, you connected applications with Services and Ingress. In this notebook, you will learn two very popular ways to manage Kubernetes YAML at scale. Helm helps you **package** an application, and Kustomize helps you **customize** an existing set of manifests.

A beginner-friendly way to think about it is this:

- **Helm** is like a package manager plus templating system for Kubernetes
- **Kustomize** is like a layering tool for YAML patches

> **Prerequisite: Notebooks 01–03.** This notebook installs into the `k8s-lab` namespace
> created in Notebook 02 and reuses the `k8s-lab/api-gateway:latest` image built there.

In [ ]:
# ── Preflight ────────────────────────────────────────────────────────────
# Every later cell shells out to these tools. Without this check a missing
# binary fails silently inside a `!` magic and you only see a confusing
# downstream error (e.g. FileNotFoundError from %%writefile) instead of
# "helm is not installed". Run this first.
import shutil
import subprocess

REQUIRED = ['kubectl', 'helm']
INSTALL_HINTS = {
    'kubectl': 'https://kubernetes.io/docs/tasks/tools/  (or `brew install kubectl`)',
    'helm': 'https://helm.sh/docs/intro/install/  (or `brew install helm`)',
}

missing = [b for b in REQUIRED if shutil.which(b) is None]
if missing:
    hint = '\n'.join(f'  - {b}: {INSTALL_HINTS[b]}' for b in missing)
    raise RuntimeError(
        f"Missing required CLI tool(s): {', '.join(missing)}\n"
        f"Install them, then re-run this cell:\n{hint}"
    )

# A reachable cluster is required too -- `kubectl` alone is not enough.
probe = subprocess.run(
    ['kubectl', 'cluster-info'], capture_output=True, text=True
)
if probe.returncode != 0:
    raise RuntimeError(
        'No reachable Kubernetes cluster. Start the one from notebook 1:\n'
        '  minikube start --cpus=4 --memory=6144 --driver=docker\n'
        f'kubectl said: {probe.stderr.strip()[:300]}'
    )

print('Preflight OK:', ', '.join(REQUIRED), '+ cluster reachable')

In [ ]:
# ── Helpers ──────────────────────────────────────────────────────────────
# Same rule as the previous notebooks: a `!helm ...` or `!kubectl ...` line that
# exits non-zero prints red text but does NOT fail the cell, so every claim made
# below is also checked in Python.
import json
import subprocess
import time

NS = "k8s-lab"


def kget(*args, ns=NS):
    cmd = ["kubectl", "get", *args, "-o", "json"] + (["-n", ns] if ns else [])
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("kubectl failed: " + r.stderr.strip()[:400])
    return json.loads(r.stdout)


def poll(predicate, timeout, interval=10):
    """Poll until predicate() is truthy; return None on timeout rather than raise.

    Used where a wait can outlast the time a notebook runner lets one cell block
    (180s is a common cap), so it is continued in the following cell."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        value = predicate()
        if value:
            return value
        time.sleep(interval)
    return None


def helm(*args):
    r = subprocess.run(["helm", *args], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"helm {' '.join(args)} failed:\n{r.stderr[-800:]}")
    return r.stdout


print("helpers ready")

## Learning Objectives

By the end of this notebook, you will be able to:

- Explain what a Helm chart is and why teams use Helm
- Add a Helm repository and install a chart into the cluster
- Inspect a Helm release and understand values and rendered manifests
- Create a simple custom Helm chart for our `api-gateway`
- Use different `values.yaml` files for dev and prod-style settings
- Render a chart locally with `helm template`
- Update a release with `helm upgrade`
- Explain what Kustomize is and when to use overlays
- Build a simple Kustomize base and a dev overlay
- Apply Kustomize resources with `kubectl apply -k`
- Compare Helm and Kustomize and know when each tool fits best

## 🛠️ Setup

Before you start:

1. Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook). If it
   doesn't appear, reload the window: `Cmd+Shift+P` → 'Reload Window'.
2. Make sure minikube is running.
3. Make sure Helm is installed on your machine (`brew install helm`).
4. Keep using the `k8s-lab` namespace from the previous notebooks.

The next cell verifies your Kubernetes context and your Helm installation.

In [ ]:
!kubectl config current-context
!kubectl get namespace k8s-lab
!helm version

## What Helm Is
Helm is the most common package manager for Kubernetes. If you have used `npm`, `pip`, or `apt`, you already understand the big idea: Helm lets you install reusable packages instead of copying and editing raw YAML by hand.

In Helm, those packages are called **charts**. A chart contains templates, default values, and metadata. When Helm installs a chart, it renders the templates into plain Kubernetes YAML and sends that YAML to the cluster.

Helm is especially helpful when:

- an app has many YAML files
- the same app needs different settings in dev and prod
- you want to reuse a deployment pattern many times

In [ ]:
!helm help | head -n 20

## Add a Helm Repository


A Helm repository is a place where charts are published. In this lab, we will add Bitnami's chart repository because it contains many well-known Kubernetes packages.

After adding the repo, we will search for the Redis chart to prove that Helm can now discover it.

> **Heads-up on Bitnami (2025 onwards).** Bitnami reorganised its public catalog: older
> tagged images were moved out of the free `docker.io/bitnami/*` namespace into
> `docker.io/bitnamilegacy/*`. A chart version that pins an image tag which is no longer
> published will install fine and then sit in `ImagePullBackOff`. If that happens to the
> Redis install below, either take the newest chart version, or retarget the image:
>
> ```bash
> helm upgrade --install my-redis bitnami/redis -n k8s-lab \
>   --set auth.enabled=false --set master.persistence.enabled=false \
>   --set global.security.allowInsecureImages=true \
>   --set image.repository=bitnamilegacy/redis
> ```
>
> This is itself a good lesson: a Helm chart is only as reproducible as the image
> registry it points at.

In [ ]:
!helm repo add bitnami https://charts.bitnami.com/bitnami --force-update
!helm repo update
!helm search repo bitnami/redis | head -n 5

## Install a Chart From a Repository
Let's install Redis from the Bitnami repo. Redis is a nice demo because it is a real application, but we can keep the setup simple by turning authentication and persistence off for the lab.

This command creates a Helm **release** named `my-redis` in the `k8s-lab` namespace.

In [ ]:
# `helm install` fails with "cannot re-use a name that is still in use" on a second
# run. `helm upgrade --install` installs if absent and upgrades if present -- the
# idempotent form, and what you want in CI too.
#
# Deliberately without `--wait`: the Redis image is a fresh download on a new
# cluster, and blocking helm until the pods are Ready can take longer than a
# notebook runner allows a single cell. Helm returns once the objects are applied;
# we wait for the pods ourselves below.
!helm upgrade --install my-redis bitnami/redis --set auth.enabled=false --set master.persistence.enabled=false -n k8s-lab --timeout 5m


def redis_ready():
    pods = kget("pods", "-l", "app.kubernetes.io/name=redis")["items"]
    ready = [p for p in pods
             if any(c["type"] == "Ready" and c["status"] == "True"
                    for c in p["status"].get("conditions", []))]
    print(f"  redis pods ready: {len(ready)}/{len(pods)}")
    return ready if pods and len(ready) == len(pods) else None


ready = redis_ready() or poll(redis_ready, timeout=160)
!kubectl get pods -n k8s-lab -l app.kubernetes.io/name=redis

# Verify, because the Bitnami image caveat above turns "installed" into
# "installed and stuck in ImagePullBackOff" with no complaint from helm.
release = json.loads(helm("list", "-n", NS, "-o", "json"))
assert any(r["name"] == "my-redis" and r["status"] == "deployed" for r in release), \
    f"my-redis is not deployed: {release}"
assert ready, (
    "the Redis chart installed but its pods never became Ready -- most likely the "
    "pinned image was moved out of the free Bitnami namespace. See the note above "
    "for the fix, or re-run this cell if it is still pulling."
)
print(f"\n✅ Helm installed a real, running application: {len(ready)} Redis pod(s) Ready")

## Inspect a Release
A Helm release stores more than just a name. Helm remembers the values you used and the manifests it rendered.

These commands answer three beginner-friendly questions:

- What releases are installed?
- What values did this release use?
- What YAML did Helm actually send to Kubernetes?

In [ ]:
!helm list -n k8s-lab
!helm get values my-redis -n k8s-lab
!helm get manifest my-redis -n k8s-lab | head -n 80

## Create Your Own Chart
Now you will create your own chart for the sample `api-gateway` service.

The original request mentioned `/tmp`, but in this lab we will use a project-local folder named `./helm-lab` so the generated files stay beside the notebook and are easier to inspect later.

`helm create` generates a starter chart with a common folder structure.

In [ ]:
!mkdir -p ./helm-lab
!rm -rf ./helm-lab/k8s-lab-chart
!helm create ./helm-lab/k8s-lab-chart
!find ./helm-lab/k8s-lab-chart -maxdepth 2 -type f | sort

## Anatomy of a Chart
A generated Helm chart usually looks like this:

```text
k8s-lab-chart/
+-- Chart.yaml
+-- values.yaml
+-- charts/
+-- templates/
    +-- deployment.yaml
    +-- service.yaml
    +-- ingress.yaml
    +-- ...
```

- `Chart.yaml` describes the chart itself
- `values.yaml` stores default configuration values
- `templates/` contains Kubernetes YAML with placeholders

In [ ]:
!find ./helm-lab/k8s-lab-chart -maxdepth 2 -type f | sort

## Customize the Chart for Our API Gateway


The starter chart contains more files than we need for a beginner lab, so we will simplify it. First we remove the extra generated templates. Then we will replace them with a minimal Deployment and Service for `api-gateway`.

This keeps the chart easy to read and easy to teach.

> **Do not forget `NOTES.txt`.** `helm create` also generates `templates/NOTES.txt` and
> `templates/_helpers.tpl`, and neither ends in `.yaml`, so `rm -f templates/*.yaml`
> leaves them behind. `NOTES.txt` is a template too — Helm renders it on every
> `install`, `upgrade` **and** `template` — and the generated one dereferences
> `.Values.ingress.enabled`. Since we are about to replace `values.yaml` with one that
> has no `ingress` key, that render fails hard with
> `nil pointer evaluating interface {}.enabled` and *nothing* installs. So the cell below
> removes `NOTES.txt` as well.

In [ ]:
!find ./helm-lab/k8s-lab-chart/templates -maxdepth 2 -type f | sort

# Remove the generated templates AND NOTES.txt. NOTES.txt is rendered on every
# install/upgrade/template and references .Values.ingress.enabled, which our
# replacement values.yaml does not define -- leaving it here makes every later
# helm command fail with "nil pointer evaluating interface {}.enabled".
!rm -f ./helm-lab/k8s-lab-chart/templates/*.yaml
!rm -f ./helm-lab/k8s-lab-chart/templates/NOTES.txt
!rm -rf ./helm-lab/k8s-lab-chart/templates/tests

print()
# _helpers.tpl only DEFINES named templates; nothing renders unless something
# calls it, so it is harmless to leave. This is what remains:
!find ./helm-lab/k8s-lab-chart -maxdepth 2 -type f | sort

In [ ]:
%%writefile ./helm-lab/k8s-lab-chart/Chart.yaml
apiVersion: v2
name: k8s-lab-chart
description: Simple Helm chart for the Kubernetes lab api-gateway
type: application
version: 0.1.0
appVersion: "1.0.0"

In [ ]:
%%writefile ./helm-lab/k8s-lab-chart/values.yaml
replicaCount: 1

image:
  repository: k8s-lab/api-gateway
  tag: latest
  pullPolicy: IfNotPresent

service:
  type: ClusterIP
  port: 8000
  targetPort: 8000

env:
  USER_SERVICE_URL: http://user-service:8001
  ORDER_SERVICE_URL: http://order-service:8002

In [ ]:
%%writefile ./helm-lab/k8s-lab-chart/templates/deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: {{ .Release.Name }}
  namespace: {{ .Release.Namespace }}
  labels:
    app: {{ .Release.Name }}
spec:
  replicas: {{ .Values.replicaCount }}
  selector:
    matchLabels:
      app: {{ .Release.Name }}
  template:
    metadata:
      labels:
        app: {{ .Release.Name }}
    spec:
      containers:
        - name: api-gateway
          image: "{{ .Values.image.repository }}:{{ .Values.image.tag }}"
          imagePullPolicy: {{ .Values.image.pullPolicy }}
          ports:
            - containerPort: {{ .Values.service.targetPort }}
          env:
            - name: USER_SERVICE_URL
              value: {{ .Values.env.USER_SERVICE_URL | quote }}
            - name: ORDER_SERVICE_URL
              value: {{ .Values.env.ORDER_SERVICE_URL | quote }}

In [ ]:
%%writefile ./helm-lab/k8s-lab-chart/templates/service.yaml
apiVersion: v1
kind: Service
metadata:
  name: {{ .Release.Name }}
  namespace: {{ .Release.Namespace }}
  labels:
    app: {{ .Release.Name }}
spec:
  type: {{ .Values.service.type }}
  selector:
    app: {{ .Release.Name }}
  ports:
    - port: {{ .Values.service.port }}
      targetPort: {{ .Values.service.targetPort }}

## Values per Environment
One of Helm's biggest strengths is that the chart stays the same while the values change. That lets you keep one chart but apply different settings for different environments.

In this lab, the dev version will use fewer replicas and a ClusterIP Service. The prod-style version will use more replicas and expose a NodePort.

In [ ]:
%%writefile ./helm-lab/k8s-lab-chart/values-dev.yaml
replicaCount: 1

service:
  type: ClusterIP
  port: 8000
  targetPort: 8000

In [ ]:
%%writefile ./helm-lab/k8s-lab-chart/values-prod.yaml
replicaCount: 3

service:
  type: NodePort
  port: 8000
  targetPort: 8000

## Render Before You Apply

`helm template` is one of the best beginner tools because it shows you the final YAML before anything is applied to the cluster. That makes debugging much easier.

In other words, Helm templates first, then Kubernetes applies the rendered result.

Three commands that look similar and are not:

| Command | Talks to the cluster? | Catches |
|---|---|---|
| `helm lint` | no | chart structure, missing `Chart.yaml` fields, obvious template errors |
| `helm template` | no | Go-template errors and the actual YAML you are about to send |
| `helm install --dry-run` | **yes** | all of the above **plus** server-side validation: unknown fields, bad `apiVersion`, admission webhooks, Pod Security violations |

`helm template` is the fast local check. `--dry-run` is the one that catches "this
`apiVersion` was removed in the Kubernetes version I am deploying to".

In [ ]:
# Pure client-side render: no cluster contact, no admission checks.
!helm template api-gateway-helm ./helm-lab/k8s-lab-chart -n k8s-lab -f ./helm-lab/k8s-lab-chart/values-dev.yaml | head -n 80

print("\n--- helm lint: catches chart-structure problems ---")
!helm lint ./helm-lab/k8s-lab-chart -f ./helm-lab/k8s-lab-chart/values-dev.yaml

# Same render, captured, so we can assert the values really drove the output.
dev = helm("template", "api-gateway-helm", "./helm-lab/k8s-lab-chart", "-n", NS,
           "-f", "./helm-lab/k8s-lab-chart/values-dev.yaml")
prod = helm("template", "api-gateway-helm", "./helm-lab/k8s-lab-chart", "-n", NS,
            "-f", "./helm-lab/k8s-lab-chart/values-prod.yaml")

assert "replicas: 1" in dev and "type: ClusterIP" in dev, \
    "values-dev.yaml did not produce 1 replica on a ClusterIP"
assert "replicas: 3" in prod and "type: NodePort" in prod, \
    "values-prod.yaml did not produce 3 replicas on a NodePort"
assert "imagePullPolicy: IfNotPresent" in dev, \
    "the chart must keep IfNotPresent or the locally built image will not be used"
print("\n✅ one chart, two values files, two different rendered manifests")

## Upgrade a Release

`helm upgrade` takes a release that already exists and applies a new chart version or new values.

In the next cell, you will:

1. install or update a dev-style release named `api-gateway-helm`
2. inspect the Deployment
3. upgrade the same release with the prod-style values
4. list the revision history and roll back to revision 1

### How Helm remembers

Helm stores the full rendered manifest of every revision in a Secret in the release's
namespace (`kubectl get secret -n k8s-lab -l owner=helm`). That is what makes
`helm history` and `helm rollback` possible, and it is a genuinely different mechanism
from `kubectl rollout undo`:

| | `kubectl rollout undo` | `helm rollback` |
|---|---|---|
| Scope | one Deployment's pod template | **every object in the release** — Services, ConfigMaps, RBAC, the lot |
| Stored in | old ReplicaSets | Helm release Secrets |
| Reverts a ConfigMap change? | no | yes, if the ConfigMap is part of the chart |

Note that `helm rollback` counts as a new revision — rolling back to revision 1 produces
revision 4, it does not delete revisions 2 and 3.

In [ ]:
!helm upgrade --install api-gateway-helm ./helm-lab/k8s-lab-chart -n k8s-lab -f ./helm-lab/k8s-lab-chart/values-dev.yaml --wait --timeout 3m
!kubectl get deploy,svc -n k8s-lab | grep api-gateway-helm

dep = kget("deployment", "api-gateway-helm")
assert dep["status"].get("readyReplicas") == 1, \
    f"dev release should be 1/1, got {dep['status'].get('readyReplicas')}"

print("\n--- upgrade to the prod-style values ---")
!helm upgrade api-gateway-helm ./helm-lab/k8s-lab-chart -n k8s-lab -f ./helm-lab/k8s-lab-chart/values-prod.yaml --wait --timeout 3m
!kubectl get deploy api-gateway-helm -n k8s-lab -o wide

dep = kget("deployment", "api-gateway-helm")
svc = kget("svc", "api-gateway-helm")
assert dep["spec"]["replicas"] == 3, f"prod values should give 3 replicas, got {dep['spec']['replicas']}"
assert svc["spec"]["type"] == "NodePort", f"prod values should give a NodePort, got {svc['spec']['type']}"

print("\n--- every upgrade is a numbered revision ---")
!helm history api-gateway-helm -n k8s-lab

print("\n--- and revisions can be rolled back ---")
!helm rollback api-gateway-helm 1 -n k8s-lab --wait --timeout 3m
!kubectl get deploy api-gateway-helm -n k8s-lab -o wide
!helm history api-gateway-helm -n k8s-lab

# The two claims made above the cell, checked:
#  (a) rollback restores EVERY object in the release, not just the Deployment
#  (b) a rollback is itself a new revision -- it does not erase 2 and 3
dep = kget("deployment", "api-gateway-helm")
svc = kget("svc", "api-gateway-helm")
assert dep["spec"]["replicas"] == 1, "rollback did not restore the Deployment"
assert svc["spec"]["type"] == "ClusterIP", \
    "rollback restored the Deployment but not the Service -- it should do both"

history = json.loads(helm("history", "api-gateway-helm", "-n", NS, "-o", "json"))
assert len(history) == 3, f"expected revisions 1, 2 and 3 to all still exist, got {len(history)}"
assert history[-1]["revision"] == 3 and "rollback" in history[-1]["description"].lower(), \
    f"the rollback should appear as revision 3, not replace revision 1: {history[-1]}"
print("\n✅ rollback restored Deployment AND Service, and became revision 3")

## What Kustomize Is
Kustomize solves a different problem from Helm. Instead of writing templates, you start with plain YAML and then apply small changes called **patches**.

This is very handy when you already have manifests and want lightweight environment overlays.

```text
Base YAML
   |
   v
+-----------+
| Overlay   |  <- small patch for dev, prod, staging
+-----------+
   |
   v
Final YAML
```

Helm is stronger when you want reusable packaged applications. Kustomize is great when you want to layer changes onto existing manifests.

In [ ]:
!kubectl kustomize --help | head -n 20

## Build a Base and an Overlay
We will build a tiny Kustomize example from scratch.

- The **base** will define a simple `api-gateway-kustomize` Deployment and Service
- The **dev overlay** will patch the Deployment to change the replica count and add a `LAB_ENV` variable

This is a very common Kustomize pattern.

In [ ]:
!mkdir -p ./kustomize-lab/base ./kustomize-lab/overlays/dev

In [ ]:
%%writefile ./kustomize-lab/base/deployment.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: api-gateway-kustomize
spec:
  replicas: 1
  selector:
    matchLabels:
      app: api-gateway-kustomize
  template:
    metadata:
      labels:
        app: api-gateway-kustomize
    spec:
      containers:
        - name: api-gateway
          image: k8s-lab/api-gateway:latest
          # Same reason as everywhere else in this lab: the image was built into
          # the cluster, and `:latest` would otherwise default the policy to Always.
          imagePullPolicy: IfNotPresent
          ports:
            - containerPort: 8000
          env:
            - name: USER_SERVICE_URL
              value: http://user-service:8001
            - name: ORDER_SERVICE_URL
              value: http://order-service:8002

In [ ]:
%%writefile ./kustomize-lab/base/service.yaml
apiVersion: v1
kind: Service
metadata:
  name: api-gateway-kustomize
spec:
  type: ClusterIP
  selector:
    app: api-gateway-kustomize
  ports:
    - port: 8000
      targetPort: 8000

In [ ]:
%%writefile ./kustomize-lab/base/kustomization.yaml
apiVersion: kustomize.config.k8s.io/v1beta1
kind: Kustomization
resources:
  - deployment.yaml
  - service.yaml

In [ ]:
%%writefile ./kustomize-lab/overlays/dev/patch.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: api-gateway-kustomize
spec:
  replicas: 2
  template:
    spec:
      containers:
        - name: api-gateway
          env:
            - name: USER_SERVICE_URL
              value: http://user-service:8001
            - name: ORDER_SERVICE_URL
              value: http://order-service:8002
            - name: LAB_ENV
              value: dev

In [ ]:
%%writefile ./kustomize-lab/overlays/dev/kustomization.yaml
apiVersion: kustomize.config.k8s.io/v1beta1
kind: Kustomization
namespace: k8s-lab
resources:
  - ../../base
patches:
  - path: patch.yaml
    target:
      kind: Deployment
      name: api-gateway-kustomize

## Apply With `kubectl apply -k`


`kubectl apply -k` tells Kubernetes to render the Kustomize overlay first and then apply the final YAML.

This is the simplest way to use Kustomize because you do not need a separate binary when `kubectl` already supports it. (`kubectl kustomize <dir>` renders without applying — the Kustomize equivalent of `helm template`.)

In [ ]:
!kubectl apply -k ./kustomize-lab/overlays/dev
!kubectl rollout status deployment/api-gateway-kustomize -n k8s-lab --timeout=180s
!kubectl get deploy,svc -n k8s-lab | grep api-gateway-kustomize

# The overlay claimed three things. Check all three on the LIVE object, not on
# the rendered YAML -- rendering correctly and applying correctly are different.
dep = kget("deployment", "api-gateway-kustomize")
assert dep["metadata"]["namespace"] == NS, \
    "the overlay's `namespace:` field should have placed this in k8s-lab"
assert dep["spec"]["replicas"] == 2, \
    f"the dev patch sets replicas: 2, live object says {dep['spec']['replicas']}"
env = {e["name"]: e.get("value") for e in dep["spec"]["template"]["spec"]["containers"][0]["env"]}
assert env.get("LAB_ENV") == "dev", f"the dev patch should add LAB_ENV=dev, got {env}"
assert dep["status"].get("readyReplicas") == 2, \
    f"only {dep['status'].get('readyReplicas')}/2 pods are ready"
print("\n✅ base + dev overlay: 2/2 ready in k8s-lab with LAB_ENV=dev")

## Compare Helm vs Kustomize


Here is a simple rule of thumb:

- Use **Helm** when you want a reusable packaged application with values and release history
- Use **Kustomize** when you already have YAML and want to patch it with overlays

The deeper difference is *when* substitution happens. Helm renders **text** — your YAML is
a Go template string until the moment it is rendered, which is why a Helm chart can build
resource names out of string concatenation but also why a mis-indented `{{ }}` produces a
YAML parse error in generated output you never wrote. Kustomize operates on **parsed
Kubernetes objects**: the base is always valid YAML you can apply on its own, and an
overlay is a structured patch. That makes Kustomize harder to abuse and less expressive —
there are no conditionals and no loops, by design.

The next cell renders both approaches so you can compare the generated output side by side.

In [ ]:
!echo "Helm render:" && helm template api-gateway-helm ./helm-lab/k8s-lab-chart -n k8s-lab -f ./helm-lab/k8s-lab-chart/values-dev.yaml | head -n 25
!echo ""
!echo "Kustomize render:" && kubectl kustomize ./kustomize-lab/overlays/dev | head -n 25

## 🧹 Clean Up
The next cell removes the demo resources created by this notebook. We keep the local chart and overlay files on disk so you can continue reading them after the lab.

In [ ]:
# `--ignore-not-found` makes uninstall idempotent (Helm 3.13+); the `|| true`
# keeps older Helm versions from failing the cell.
!helm uninstall my-redis -n k8s-lab --ignore-not-found || true
!helm uninstall api-gateway-helm -n k8s-lab --ignore-not-found || true
!kubectl delete -k ./kustomize-lab/overlays/dev --ignore-not-found

print()
# `helm uninstall` does NOT delete PersistentVolumeClaims created by a chart's
# StatefulSet volumeClaimTemplates -- Kubernetes keeps them on purpose so an
# accidental delete does not destroy your data. The Redis chart's replicas leave
# three 8Gi claims behind, and on a cloud provider those are three disks you keep
# paying for. Notebook 09 covers the mechanism; here we just tidy up.
!kubectl get pvc -n k8s-lab -l app.kubernetes.io/instance=my-redis
!kubectl delete pvc -n k8s-lab -l app.kubernetes.io/instance=my-redis --ignore-not-found

print()
!helm list -n k8s-lab

# Nothing from this notebook should still be running.
assert json.loads(helm("list", "-n", NS, "-o", "json")) == [], \
    "a Helm release from this notebook is still installed"
orphans = [c["metadata"]["name"] for c in kget("pvc")["items"]
           if "my-redis" in c["metadata"]["name"]]
assert not orphans, f"the chart's PVCs were not removed: {orphans}"
print("\n✅ releases uninstalled and their orphaned PVCs removed")

## 🎓 What You Learned
In this notebook, you learned that:

- **Helm** packages Kubernetes applications into reusable charts
- A Helm **release** is an installed chart with its own values and history
- `helm repo add`, `helm install`, `helm get`, `helm template`, and `helm upgrade` are core daily commands
- A custom chart can stay small and beginner-friendly by keeping only the templates you need
- Different values files let one chart behave differently in dev and prod-style environments
- **Kustomize** works by starting with base YAML and layering patches on top
- `kubectl apply -k` is the easiest way to use Kustomize in everyday workflows
- Helm and Kustomize both solve configuration problems, but they do it in different ways

If these ideas feel comfortable now, you are ready for the next stage of the lab series where Kubernetes becomes easier to operate at scale.